# Third Notebook to use

Todo: Remove fluff, make sure it also works with unseen_movies. Then copy paste in encoding from final_db.ipynb

In [2]:
import pandas as pd
import re
import unicodedata
from sklearn.preprocessing import MultiLabelBinarizer
import ast

In [3]:
df_director_info = pd.read_csv('directors.csv')

In [31]:
import ast
from typing import List
def cat_to_list(df: pd.DataFrame, columnnames: List[str]):
    for column in columnnames:
        if type(df.loc[0,column]) == str:
            df[column] = df[column].apply(lambda x: ast.literal_eval(x))
    return df

## Now they are on the same form. Time to make a function that adds director stats! Copied from final_db but into a function rather than just individual cells.

In [32]:
df_director_info = pd.read_csv('directors.csv')
def half_column_value(val):
    return val/2
df_director_info['director_avg_movie_score'] = df_director_info['director_avg_movie_score'].apply(half_column_value)
df_director_info['director_avg_last_5_movies'] = df_director_info['director_avg_last_5_movies'].apply(half_column_value)
df_director_info.head()

,director,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies
0,Steven Spielberg,42,3.495619,3.54200
1,Sidney Lumet,50,3.189030,3.20550
2,Hayao Miyazaki,39,3.619756,3.74060
3,Aditya Chopra,4,3.470250,3.47025
4,Frank Darabont,7,3.607571,3.85060


In [35]:
def add_direc_stats(df_data: pd.DataFrame, df_director_stats: pd.DataFrame):
    """
    df_data is the original data, while df_director_stats has the stats of the 
    directors included in df_data that you want to add to the dataset. 
    df_director_stats scores are assumed to have been normalized to be out of 5.
    """
    df_data = cat_to_list(df_data, ['director'])
    df_exploded = df_data.explode('director')
    from useful_funcs import normalize_title
    df_exploded['director'] = df_exploded['director'].apply(normalize_title)
    df_director_stats['director'] = df_director_stats['director'].apply(normalize_title)
    df_merged = df_exploded.merge(
        df_director_info,
        left_on='director',
        right_on='director',
        how='left'
    )
    # only drop if there are na values in one of the imported rows
    df_clean = df_merged.dropna(subset='director_avg_num_movies')
    mean_cols = [                       # the columns we want to average
        "director_avg_num_movies",
        "director_avg_movie_score",
        "director_avg_last_5_movies"
    ]
    # combine columns either with mean or 'first' which is just combining cols with 2 same rows
    other_cols = [c for c in df_clean.columns if c not in mean_cols]
    agg_dict = {c: 'first' for c in other_cols}
    agg_dict.update({c: 'mean' for c in mean_cols})
    agg_dict['director'] = lambda x: list(dict.fromkeys(x))     # list from dict to ensure uniqueness
    df_avg = df_clean.groupby('title', as_index=False).agg(agg_dict)

    # make director 1 and director 2 since we limited to 2 directors
    df_avg["director1"] = df_avg["director"].apply(lambda x: x[0] if len(x) > 0 else None)
    df_avg["director2"] = df_avg["director"].apply(lambda x: x[1] if len(x) > 1 else None)

    df_avg.drop('director', axis=1, inplace=True)

    return df_avg
    
    

In [38]:
df_training = pd.read_csv('initial_dframe.csv')
df_training = add_direc_stats(df_training, df_director_info)

In [39]:
df_unseen = pd.read_csv('unseen_movies.csv')
df_unseen = add_direc_stats(df_unseen, df_director_info)

In [41]:
df_training.to_csv('initial_dframe_direc_stats.csv', index=False)
df_unseen.to_csv('unseen_movies_direc_stats.csv', index=False)